In [0]:
CATALOG = "workspace"
SOURCE = f"/Volumes/{CATALOG}/landing/raw/entsoe/day_ahead_prices/"
TARGET = f"{CATALOG}.bronze.entsoe_day_ahead_prices"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/entsoe_day_ahead_prices" # A checkpoint is Auto Loader’s memory: a folder where it records which files it has already processed and how far each run got

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.checkpoints")

In [0]:
import os, sys
print("Notebook folder:", os.getcwd())
print("Files here:", os.listdir("."))
print("Folder on path:", os.getcwd() in sys.path)

In [0]:
from pyspark.sql import functions as F, types as T
from entsoe_parser import parse_day_ahead

point_schema = T.ArrayType(T.StructType([
    T.StructField("delivery_start_utc", T.TimestampType()),
    T.StructField("price", T.DoubleType()),
    T.StructField("currency", T.StringType()),
    T.StructField("price_unit", T.StringType()),
    T.StructField("resolution", T.StringType()),
    T.StructField("position", T.IntegerType()),
    T.StructField("period_start_utc", T.TimestampType()),
    T.StructField("curve_type", T.StringType()),
]))

@F.udf(returnType=point_schema) # wrapping the parser for Spark
def parse_udf(content):
    return parse_day_ahead(bytes(content)) if content else []

In [0]:
raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")        # read each file whole
    .option("pathGlobFilter", "*.xml")
    .option("cloudFiles.allowOverwrites", "true")     # re-read files the Lambda overwrites (revisions)
    .load(SOURCE)
) # where to read

bronze = (
    raw
    .withColumn("point", F.explode(parse_udf("content")))
    .select(
        F.regexp_extract("path", r"bzn=([^/]+)/", 1).alias("bzn"),
        F.to_date(F.regexp_extract("path", r"delivery_date=(\d{4}-\d{2}-\d{2})", 1)).alias("delivery_date"),
        "point.*",
        F.col("path").alias("_source_file"),
        F.col("modificationTime").alias("_file_modified_at"),
        F.current_timestamp().alias("_ingested_at"),
    )
)

query = (
    bronze.writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)      # process what's new, then stop
    .toTable(TARGET)
)
query.awaitTermination()